# 📅 Week 4: Transfer Learning — Theory & Baseline Experiment
## TMI-102: Finetuning Deep Learning Models on Astrophysical Datasets
### Physics and Astronomy Club (PaAC), IIT Roorkee

---

**Objective:** Understand why ImageNet-pretrained features transfer to astrophysics, implement feature extraction with a frozen ResNet-50 backbone, and establish the baseline accuracy.

**Key Question:** *Why would a model trained on cats, cars, and furniture be useful for classifying galaxies 100 million light-years away?*

---

### 📚 Required Reading
| Paper | Link |
|:---|:---|
| Yosinski et al. (2014) — How Transferable are Features? | [arXiv:1411.1792](https://arxiv.org/abs/1411.1792) |
| CS231n: Transfer Learning | [Notes](https://cs231n.github.io/transfer-learning/) |
| TF/Keras Transfer Learning Tutorial | [TensorFlow Guide](https://www.tensorflow.org/tutorials/images/transfer_learning) |

### 🎯 Deliverables
- [ ] Explain the feature hierarchy in deep CNNs
- [ ] Load pretrained ResNet-50 and freeze all layers
- [ ] Train a classifier head on Galaxy Zoo
- [ ] Report baseline accuracy and analyze failure modes
- [ ] Answer all conceptual questions in markdown cells

## Part 1: Conceptual Questions (30 pts)

**Q1.** In a deep CNN like ResNet-50, the early layers learn features like edges and textures, while deeper layers learn complex shapes and object parts. Explain why this **feature hierarchy** might transfer from ImageNet (natural images) to galaxy morphology classification.

*Your answer:*
CNNs trained on ImageNet learn universal visual building blocks that apply to any optical image, including galaxies:

Early layers detect universal primitives like edges, contrast gradients, and simple textures. The math of a sharp edge is the same whether it is a building on Earth or a bright galactic core against dark space.

Middle layers combine these edges into broader geometric shapes like circles and sweeping curves—structures perfectly suited for identifying elliptical halos or spiral arms.

Deep layers recombine these shapes into complex objects.

When transferring the model to astronomy, we reuse the early and middle layers because the model already knows "how to see" geometry. We only need to retrain the final layers to map those familiar curves and textures to galaxy types instead of everyday objects.


**Q2.** What is the difference between **feature extraction** (frozen backbone) and **fine-tuning** (unfreezing layers)? When would you use each?

*Your answer:*

Feature Extraction (Frozen Backbone)
This is when you "freeze" the entire pre-trained model so its weights can't change. You just add a new, untrained classifier layer to the very end and only train that one layer.

When to use:I'd use this if my dataset is really small (to avoid overfitting) or if my images are very similar to what the model originally trained on (like taking a model trained on general ImageNet and using it to classify specific breeds of cats).

Fine-Tuning (Unfreezing Layers)
This is when you "unfreeze" some or all of the layers in the pre-trained model and actually update their weights during training so they adapt to the new data.

When to use:I'd use this if I have a large dataset (so it's safe to update millions of parameters without memorizing the data) or if my images are totally different from the original data (like transferring from everyday photos to medical X-rays or telescope images), meaning the model has to learn brand new complex features.

**Q3.** The "domain gap" between ImageNet and Galaxy Zoo is significant (terrestrial vs. astronomical). List 3 specific low-level features that would transfer well and 2 that would NOT.

*Your answer:*
3 Features That Transfer Well

Edges and Contours: The sharp contrast between a bright object and a dark background is mathematically identical whether it is a building against the sky or a galaxy against the vacuum of space.

Point-like Blobs: Filters that detect localized bright spots perfectly map to identifying dense galactic cores or foreground stars.

Smooth Gradients: The gradual fading from light to dark (intensity gradients) transfers seamlessly to detecting the diffuse, fuzzy halos of elliptical galaxies.

2 Features That Do NOT Transfer Well

Color Relationships: ImageNet filters learn specific Earth-based color correlations (like green leaves or blue skies). Galaxy images often map non-visible spectrums (like radio or infrared) to RGB channels, rendering terrestrial color detectors useless.

Directional Shadows: Natural images rely heavily on cast shadows and directional sunlight to infer 3D shape and depth. Galaxies are luminous bodies that emit their own light, so filters tuned to look for cast shadows will not find useful patterns.

## Part 2: Setup & Data Loading (10 pts)

Load the Galaxy Zoo dataset and create data generators. Use the same preprocessing from Week 3.

In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import splitfolders

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 3
CLASS_NAMES = ['Spiral', 'Elliptical', 'Merger']


input_folder = 'C:/Users/Lenovo/Documents/pranay 1/python 1.0/Galaxy_Dataset/images_training_rev1'
output_folder = 'C:/Users/Lenovo/Documents/pranay 1/python 1.0/Galaxy_Dataset/dataset_split'

splitfolders.ratio(input_folder, output=output_folder, seed=42, ratio=(0.7, 0.15, 0.15))

train_dir = os.path.join(output_folder, 'train')
val_dir = os.path.join(output_folder, 'val')
test_dir = os.path.join(output_folder, 'test')

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=360,
    horizontal_flip=True,
    vertical_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255) 


train_generator = train_datagen.flow_from_directory(
    train_dir, 
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_dir,    
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    test_dir,   
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

TensorFlow version: 2.22.0-rc0
GPU available: False
Found 43104 images belonging to 1 classes.
Found 9236 images belonging to 1 classes.
Found 9238 images belonging to 1 classes.
